In [ ]:
import pandas as pd
import numpy as np
import pickle
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()

server = os.getenv('SQL_SERVER')
database = os.getenv('SQL_DATABASE')

connection_string = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(connection_string)

try:
    with engine.connect() as conn:
        print(f"Conectado com sucesso ao banco: {database}")
except Exception as e:
    print(f"Erro na conexão: {e}")

In [ ]:
# Carregar o modelo treinado
with open('modelo_credito.pkl', 'rb') as f:
    modelo = pickle.load(f)

print("Modelo carregado com sucesso!")

In [ ]:
# Hard Rules - regras rígidas de negação
def aplicar_hard_rules(cliente):
    
    # Regra 1: possui restrição ativa
    if cliente['possui_restricao'] == 1:
        return False, "Negado - possui restrição ativa"
    
    # Regra 2: mais de 2 empréstimos ativos
    if cliente['qtd_emprestimos_ativos'] > 2:
        return False, "Negado - mais de 2 empréstimos ativos"
    
    # Regra 3: comprometimento de renda acima de 30%
    parcela_estimada = cliente['valor_solicitado'] / cliente['prazo_meses']
    comprometimento = parcela_estimada / cliente['renda_mensal']
    if comprometimento > 0.30:
        return False, f"Negado - comprometimento de renda em {comprometimento:.1%}"
    
    return True, "Passou nas regras"

print("Hard rules definidas com sucesso!")

In [ ]:
# Função principal de decisão
def decidir_credito(cliente):
    
    # Etapa 1: verificar hard rules
    aprovado, motivo = aplicar_hard_rules(cliente)
    if not aprovado:
        return {'decisao': 'NEGADO', 'motivo': motivo, 'probabilidade_risco': None}
    
    # Etapa 2: calcular score do modelo ML
    colunas = ['idade', 'renda_mensal', 'score_credito', 'tem_imovel', 'tem_veiculo',
               'tempo_emprego_anos', 'qtd_emprestimos_ativos', 'historico_inadimplencia',
               'possui_restricao', 'valor_solicitado', 'prazo_meses']
    
    dados = pd.DataFrame([cliente])[colunas]
    probabilidade_risco = modelo.predict_proba(dados)[0][1]
    
    # Etapa 3: aplicar faixas de decisão
    if probabilidade_risco <= 0.40:
        decisao = 'APROVADO'
        motivo = f"Risco baixo - aprovado automaticamente"
    elif probabilidade_risco <= 0.70:
        decisao = 'REVISAO'
        motivo = f"Risco moderado - requer análise manual"
    else:
        decisao = 'NEGADO'
        motivo = f"Risco alto - negado automaticamente"
    
    return {
        'decisao': decisao,
        'motivo': motivo,
        'probabilidade_risco': f"{probabilidade_risco:.1%}"
    }

print("Função de decisão definida com sucesso!")

In [ ]:
# Carregar clientes do banco para testar
query = "SELECT TOP 10 * FROM clientes_v2"
df_teste = pd.read_sql(query, engine)

# Aplicar o sistema de decisão para cada cliente
resultados = []
for _, cliente in df_teste.iterrows():
    resultado = decidir_credito(cliente)
    resultado['nome'] = cliente['nome']
    resultados.append(resultado)

df_resultados = pd.DataFrame(resultados)[['nome', 'decisao', 'motivo', 'probabilidade_risco']]
print(df_resultados.to_string())